# Create Interactive Webpage - load DuckDB file

**a standalone interactive HTML file** that a user can open locally in their browser, select a DuckDB file from their own hard drive, and explore the data without needing Google Colab's backend or a running Python server.

To do this, we have to swap the Python backend for DuckDB-Wasm. This allows the browser to run the DuckDB engine internally using WebAssembly.

The Problem with your current HTML
Your current code uses google.colab.kernel.invokeFunction, which only works inside a Colab notebook. To make it "run on any computer," we need to:

1. Include the DuckDB-Wasm libraries via CDN.

2. Replace the file selection logic to use a standard <input type="file">.

3. Replace the Python filtering with browser-side SQL queries.

In [ ]:
%%html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>DuckDB Submittal Explorer</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body { background-color: #f4f7f6; padding: 20px; font-family: system-ui, -apple-system, sans-serif; }
        .main-card { background: white; padding: 25px; border-radius: 12px; box-shadow: 0 8px 30px rgba(0,0,0,0.05); }
        .filter-bar { background: #f8f9fa; padding: 15px; border-radius: 8px; margin-bottom: 20px; border: 1px solid #eee; }
        .table-container { max-height: 600px; overflow-y: auto; border: 1px solid #dee2e6; border-radius: 4px; }
        th { position: sticky; top: 0; background: #fff !important; z-index: 2; box-shadow: inset 0 -2px 0 #dee2e6; }
        .status-pill { padding: 4px 10px; border-radius: 20px; font-size: 0.75rem; font-weight: 600; }
        #loadingOverlay { display: none; position: fixed; top: 0; left: 0; width: 100%; height: 100%;
                          background: rgba(255,255,255,0.8); z-index: 9999; justify-content: center; align-items: center; flex-direction: column; }
    </style>
</head>
<body>

<div id="loadingOverlay">
    <div class="spinner-border text-primary" role="status"></div>
    <div class="mt-3 fw-bold" id="loadingMessage">Initializing Database...</div>
</div>

<div class="container-fluid">
    <div class="main-card">
        <h2 class="mb-4">Submittal Data Explorer</h2>

        <div class="row filter-bar g-3">
            <div class="col-md-3">
                <label class="form-label small fw-bold">1. LOAD FILE</label>
                <input type="file" id="filePicker" class="form-control form-control-sm" accept=".duckdb">
            </div>
            <div class="col-md-3">
                <label class="form-label small fw-bold">SEARCH</label>
                <input type="text" id="searchInput" class="form-control form-control-sm" placeholder="Search any column...">
            </div>
            <div class="col-md-3">
                <label class="form-label small fw-bold">DISCIPLINE</label>
                <select id="discFilter" class="form-select form-select-sm"><option value="">All</option></select>
            </div>
            <div class="col-md-3 d-flex align-items-end">
                <button id="downloadBtn" class="btn btn-sm btn-success w-100" disabled>Export CSV</button>
            </div>
        </div>

        <div class="table-container">
            <table class="table table-hover table-sm align-middle">
                <thead id="tblHead"></thead>
                <tbody id="tblBody">
                    <tr><td colspan="100%" class="text-center p-5 text-muted">Please select a .duckdb file to start.</td></tr>
                </tbody>
            </table>
        </div>
    </div>
</div>

<script type="module">
    // Using unpkg as a reliable CDN source
    import * as duckdb from 'https://unpkg.com/@duckdb/duckdb-wasm@1.28.0/dist/duckdb-wasm-browser.js';

    let db, conn, tableName;
    const overlay = document.getElementById('loadingOverlay');

    // 1. Boot the SQL Engine
    async function startEngine() {
        overlay.style.display = 'flex';
        try {
            const MANUAL_BUNDLES = {
                mainModule: 'https://unpkg.com/@duckdb/duckdb-wasm@1.28.0/dist/duckdb-mvp.wasm',
                mainWorker: 'https://unpkg.com/@duckdb/duckdb-wasm@1.28.0/dist/duckdb-browser-mvp.worker.js',
            };
            const worker = new Worker(MANUAL_BUNDLES.mainWorker);
            db = new duckdb.AsyncDuckDB(new duckdb.ConsoleLogger(), worker);
            await db.instantiate(MANUAL_BUNDLES.mainModule);
            overlay.style.display = 'none';
        } catch (e) {
            document.getElementById('loadingMessage').innerHTML = `<span class="text-danger">Connection Blocked by Firewall/VPN.</span>`;
            console.error(e);
        }
    }

    // 2. Handle File Selection
    document.getElementById('filePicker').addEventListener('change', async (e) => {
        const file = e.target.files[0];
        if (!file) return;

        overlay.style.display = 'flex';
        document.getElementById('loadingMessage').innerText = "Reading Database...";

        try {
            await db.registerFileHandle('data.duckdb', file, duckdb.DuckDBDataProtocol.BROWSER_FILEREADER, true);
            conn = await db.connect();

            // Auto-detect the table name
            const tables = await conn.query("SHOW TABLES");
            if (tables.toArray().length === 0) throw new Error("No tables found in file.");
            tableName = tables.toArray()[0].name;

            await updateFilters();
            await runDataQuery();
            document.getElementById('downloadBtn').disabled = false;
        } catch (err) {
            alert("Error: " + err.message);
        } finally {
            overlay.style.display = 'none';
        }
    });

    // 3. Populate Unique Filters
    async function updateFilters() {
        const res = await conn.query(`SELECT DISTINCT Discipline FROM "${tableName}" WHERE Discipline IS NOT NULL ORDER BY 1`);
        const select = document.getElementById('discFilter');
        select.innerHTML = '<option value="">All Disciplines</option>';
        res.toArray().forEach(row => {
            select.innerHTML += `<option value="${row.Discipline}">${row.Discipline}</option>`;
        });
    }

    // 4. Run SQL Query
    async function runDataQuery() {
        if (!conn) return;
        const search = document.getElementById('searchInput').value;
        const disc = document.getElementById('discFilter').value;

        let sql = `SELECT * FROM "${tableName}" WHERE 1=1`;
        if (disc) sql += ` AND Discipline = '${disc}'`;
        if (search) {
            // Searches all columns by casting to text
            sql += ` AND (CAST( (SELECT t FROM "${tableName}" t WHERE t."Doc ID" = "${tableName}"."Doc ID") AS TEXT) ILIKE '%${search}%')`;
        }
        sql += ` LIMIT 500`;

        const result = await conn.query(sql);
        renderTable(result.toArray());
    }

    function renderTable(data) {
        const head = document.getElementById('tblHead');
        const body = document.getElementById('tblBody');
        head.innerHTML = ''; body.innerHTML = '';
        if (data.length === 0) return;

        const cols = Object.keys(data[0]);
        let hRow = '<tr>';
        cols.forEach(c => hRow += `<th>${c}</th>`);
        head.innerHTML = hRow + '</tr>';

        data.forEach(row => {
            let bRow = '<tr>';
            cols.forEach(c => bRow += `<td>${row[c] ?? ''}</td>`);
            body.innerHTML += bRow + '</tr>';
        });
    }

    // Listeners
    document.getElementById('searchInput').addEventListener('input', runDataQuery);
    document.getElementById('discFilter').addEventListener('change', runDataQuery);

    startEngine();
</script>

</body>
</html>


# Use Colab to create interactive page - Load DuckDB file




In [ ]:
import duckdb
import pandas as pd
import glob
import json
from google.colab import output

# Global variable to hold the dataframe for filtering
current_df = pd.DataFrame()

def list_duckdb_files():
    # Looks for .duckdb files in the root folder /content/
    files = glob.glob("*.duckdb")
    print(f"Backend found files: {files}") # This prints in the Python cell logs
    return json.dumps(files)

def load_duckdb_data(filename):
    global current_df
    try:
        conn = duckdb.connect(filename)
        # Automatically gets the first table name found in the file
        tables = conn.execute("SHOW TABLES").fetchall()
        if not tables:
            return json.dumps([{"Error": "No tables found in file"}])

        table_name = tables[0][0]
        current_df = conn.execute(f"SELECT * FROM {table_name}").df()
        conn.close()

        # Convert any datetime columns to strings for JSON serialization
        for col in current_df.select_dtypes(include=['datetime64']).columns:
            current_df[col] = current_df[col].dt.strftime('%Y-%m-%d')

        return current_df.to_json(orient='records')
    except Exception as e:
        return json.dumps([{"Error": str(e)}])

def filter_data(global_search, doc_id, discipline, approval, closed, overdue, sort_col, sort_dir):
    global current_df
    if current_df.empty: return json.dumps([])

    df = current_df.copy()
    # Apply filtering logic here (as seen in previous steps)
    # ...
    return df.to_json(orient='records')

# IMPORTANT: These must be registered for the JS to "see" them
output.register_callback('list_duckdb_files', list_duckdb_files)
output.register_callback('load_duckdb_data', load_duckdb_data)
output.register_callback('filter_data', filter_data)

In [ ]:
import json
from IPython.display import HTML

# HTML and JavaScript template for the interactive webpage
HTML_TEMPLATE = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Interactive Data Table</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body {{ font-family: 'Segoe UI', sans-serif; margin: 20px; background: #f0f2f5; color: #333; }}
        .container-fluid {{ padding: 20px; background: white; border-radius: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.1); }}
        h1 {{ color: #007bff; margin-bottom: 20px; text-align: center; }}
        .filters, .pagination-controls {{ display: flex; flex-wrap: wrap; gap: 10px; margin-bottom: 20px; align-items: center; }}
        .filter-group {{ display: flex; flex-direction: column; min-width: 150px; }}
        .filter-group label {{ margin-bottom: 5px; font-weight: bold; color: #555; }}
        input[type="text"], select {{ padding: 8px 12px; border: 1px solid #ced4da; border-radius: 5px; font-size: 1rem; width: 100%; box-sizing: border-box; transition: border-color 0.2s; }}
        input[type="text"]:focus, select:focus {{ border-color: #80bdff; outline: 0; box-shadow: 0 0 0 0.2rem rgba(0,123,255,.25); }}
        .table-responsive {{ overflow-x: auto; }}
        .table-striped tbody tr:nth-of-type(odd) {{ background-color: rgba(0,0,0,.05); }}
        .table-hover tbody tr:hover {{ background-color: rgba(0,0,0,.075); cursor: pointer; }}
        th {{ cursor: pointer; white-space: nowrap; padding: 12px 15px; background-color: #e9ecef; border-bottom: 2px solid #dee2e6; color: #495057; font-weight: 600; text-align: left; }}
        td {{ padding: 10px 15px; border-top: 1px solid #dee2e6; vertical-align: middle; }}
        .sort-arrow {{ margin-left: 5px; visibility: hidden; }}
        .sort-asc .sort-arrow.asc, .sort-desc .sort-arrow.desc {{ visibility: visible; }}
        .pagination-controls button {{ background-color: #007bff; color: white; border: none; padding: 8px 15px; border-radius: 5px; cursor: pointer; font-size: 0.9rem; transition: background-color 0.2s; }}
        .pagination-controls button:hover:not(:disabled) {{ background-color: #0056b3; }}
        .pagination-controls button:disabled {{ background-color: #a0cfee; cursor: not-allowed; }}
        .page-info {{ margin: 0 10px; font-weight: bold; color: #555; }}
        #noDataMessage {{ text-align: center; color: #dc3545; font-size: 1.2rem; margin-top: 20px; display: none; }}
    </style>
</head>
<body>
    <div class="container-fluid">
        <h1>Processed Submittal Tracker</h1>

        <div class="filters">
            <div class="filter-group">
                <label for="duckdbFileSelect">Select DuckDB File:</label>
                <select id="duckdbFileSelect">
                    <option value="">Loading files...</option>
                </select>
            </div>
            <div class="filter-group">
                <label for="docIdFilter">Doc ID:</label>
                <select id="docIdFilter">
                    <option value="">All</option>
                </select>
            </div>
            <div class="filter-group">
                <label for="disciplineFilter">Discipline:</label>
                <select id="disciplineFilter">
                    <option value="">All</option>
                </select>
            </div>
            <div class="filter-group">
                <label for="itemsPerPage">Items per page:</label>
                <select id="itemsPerPage">
                    <option value="10">10</option>
                    <option value="25">25</option>
                    <option value="50">50</option>
                    <option value="100">100</option>
                    <option value="-1">All</option>
                </select>
            </div>
            <div class="filter-group">
                <label for="globalFilter">Search All Columns:</label>
                <input type="text" id="globalFilter" placeholder="Type to search...">
            </div>
            <div class="filter-group">
                <label for="approvalStatusFilter">Approval Status:</label>
                <select id="approvalStatusFilter">
                    <option value="">All</option>
                </select>
            </div>
            <div class="filter-group">
                <label for="closedFilter">Closed:</label>
                <select id="closedFilter">
                    <option value="">All</option>
                    <option value="YES">YES</option>
                    <option value="NO">NO</option>
                </select>
            </div>
            <div class="filter-group">
                <label for="overdueFilter">Overdue to Resubmit:</label>
                <select id="overdueFilter">
                    <option value="">All</option>
                    <option value="Resubmitted">Resubmitted</option>
                    <option value="Overdue">Overdue</option>
                    <option value="Not Due">Not Due</option>
                    <option value="N/A">N/A</option>
                </select>
            </div>
        </div>

        <div class="table-responsive">
            <table class="table table-striped table-hover">
                <thead>
                    <tr id="tableHeader"></tr>
                </thead>
                <tbody id="tableBody"></tbody>
            </table>
        </div>
        <p id="noDataMessage">No data matches your current filters.</p>

        <div class="pagination-controls justify-content-center">
            <button id="prevPage" disabled>&laquo; Previous</button>
            <span class="page-info">Page <span id="currentPage">1</span> of <span id="totalPages">1</span></span>
            <button id="nextPage" disabled>Next &raquo;</button>
        </div>
    </div>

    <script>
        let originalData = []; // Data will be loaded dynamically
        let filteredData = [];
        let currentSortColumn = null;
        let currentSortDirection = 'asc';
        let currentPage = 1;
        let itemsPerPage = parseInt(document.getElementById('itemsPerPage').value);

        const tableBody = document.getElementById('tableBody');
        const tableHeader = document.getElementById('tableHeader');
        const globalFilter = document.getElementById('globalFilter');
        const docIdFilter = document.getElementById('docIdFilter');
        const disciplineFilter = document.getElementById('disciplineFilter');
        const approvalStatusFilter = document.getElementById('approvalStatusFilter');
        const closedFilter = document.getElementById('closedFilter');
        const overdueFilter = document.getElementById('overdueFilter');
        const itemsPerPageSelect = document.getElementById('itemsPerPage');
        const duckdbFileSelect = document.getElementById('duckdbFileSelect'); // New dropdown

        const prevPageBtn = document.getElementById('prevPage');
        const nextPageBtn = document.getElementById('nextPage');
        const currentPageSpan = document.getElementById('currentPage');
        const totalPagesSpan = document.getElementById('totalPages');
        const noDataMessage = document.getElementById('noDataMessage');

        // NEW HELPER FUNCTION to reset UI after data load
        async function resetUIWithData(newData) {{
            originalData = newData;
            filteredData = [...originalData];
            currentSortColumn = null; // Reset sort state
            currentSortDirection = 'asc';
            currentPage = 1;

            initializeTableHeaders(); // Re-initialize headers based on new data
            populateFilters();        // Re-populate filters based on new data
            await applyFilters();     // Re-apply filters and render table
        }}

        // Asynchronously load initial data and populate filters
        async function initializePage() {{
            await listDuckDBFiles(); // List available files
            // Optionally, load a default file or wait for user selection
            if (duckdbFileSelect.options.length > 1) {{
                duckdbFileSelect.selectedIndex = 1; // Select the first actual file
                await loadSelectedDuckDBFile();
            }}
            else {{
                 console.log("No DuckDB files found or loaded initially."); // More specific log
                 await resetUIWithData([]); // If no files, reset UI with empty data
            }}
            // Removed redundant calls to initializeTableHeaders(), populateFilters(), applyFilters()
            // as they are handled by resetUIWithData
        }}

        // New functions for dynamic file loading
        async function listDuckDBFiles() {{
            console.log('Listing DuckDB files via callback...');
            const files = JSON.parse(await google.colab.kernel.invokeFunction('list_duckdb_files', []));
            console.log('Files received from backend:', files);
            duckdbFileSelect.innerHTML = '<option value="">Select File</option>'; // Reset dropdown
            if (files && files.length > 0) {{
                console.log('Populating dropdown with files...');
                files.forEach(file => {{
                    const option = document.createElement('option');
                    option.value = file;
                    option.textContent = file;
                    duckdbFileSelect.appendChild(option);
                }});
                console.log('Dropdown options after population:', duckdbFileSelect.options.length);
            }} else {{
                console.log('No files to populate dropdown, or files is empty/null.');
                const option = document.createElement('option');
                option.value = "";
                option.textContent = "No .duckdb files found";
                duckdbFileSelect.appendChild(option);
                duckdbFileSelect.disabled = true;
            }}
            console.log('DuckDB files listed:', files);
        }}

        async function loadSelectedDuckDBFile() {{
            const filename = duckdbFileSelect.value;
            if (!filename) {{
                console.log('No file selected, resetting UI.');
                await resetUIWithData([]); // Reset with empty data if no file is selected
                return;
            }}
            // FIX: Replace JavaScript template literal with string concatenation to avoid Python f-string parsing
            console.log('Loading data from ' + filename + ' via callback...');
            const jsonData = await google.colab.kernel.invokeFunction('load_duckdb_data', [filename]);
            const newData = JSON.parse(jsonData); // Parse new data
            console.log('Loaded ' + newData.length + ' records from ' + filename + '.');
            await resetUIWithData(newData); // Call the helper to reset UI with new data
        }}


        // Initialize Headers
        function initializeTableHeaders() {{
            if (originalData.length === 0) {{
                tableHeader.innerHTML = '';
                return;
            }}
            const columns = Object.keys(originalData[0]);
            tableHeader.innerHTML = '';
            columns.forEach(col => {{
                const th = document.createElement('th');
                th.textContent = col;
                th.setAttribute('data-column', col);
                th.innerHTML = col + ' <span class="sort-arrow asc">&#9650;</span><span class="sort-arrow desc">&#9660;</span>';
                th.addEventListener('click', () => sortTable(col));
                tableHeader.appendChild(th);
            }});
        }}

        // Populate filter options
        function populateFilters() {{
            if (originalData.length === 0) {{
                docIdFilter.innerHTML = '<option value="">All</option>';
                disciplineFilter.innerHTML = '<option value="">All</option>';
                approvalStatusFilter.innerHTML = '<option value="">All</option>';
                overdueFilter.innerHTML = '<option value="">All</option>';
                return;
            }}

            // Populate Doc ID filter
            const docIds = [...new Set(originalData.map(item => item['Doc ID']))].sort();
            docIdFilter.innerHTML = '<option value="">All</option>';
            docIds.forEach(id => {{
                if (id) {{
                    const option = document.createElement('option');
                    option.value = id;
                    option.textContent = id;
                    docIdFilter.appendChild(option);
                }}
            }});

            // Populate Discipline filter
            const disciplines = [...new Set(originalData.map(item => item['Discipline']))].sort();
            disciplineFilter.innerHTML = '<option value="">All</option>';
            disciplines.forEach(discipline => {{
                if (discipline) {{
                    const option = document.createElement('option');
                    option.value = discipline;
                    option.textContent = discipline;
                    disciplineFilter.appendChild(option);
                }}
            }});

            const approvalStatuses = [...new Set(originalData.map(item => item['Latest Approval Status']))].sort();
            approvalStatusFilter.innerHTML = '<option value="">All</option>';
            approvalStatuses.forEach(status => {{
                if (status) {{
                    const option = document.createElement('option');
                    option.value = status;
                    option.textContent = status;
                    approvalStatusFilter.appendChild(option);
                }}
            }});

            const overdueStatuses = [...new Set(originalData.map(item => item['Overdue to resubmit']))].sort();
            overdueFilter.innerHTML = '<option value="">All</option>';
            overdueStatuses.forEach(status => {{
                if (status) {{
                    const option = document.createElement('option');
                    option.value = status;
                    option.textContent = status;
                    overdueFilter.appendChild(option);
                }}
            }});
        }}

        async function renderTable() {{
            tableBody.innerHTML = '';
            noDataMessage.style.display = 'none';

            if (filteredData.length === 0) {{
                noDataMessage.style.display = 'block';
                currentPageSpan.textContent = 0;
                totalPagesSpan.textContent = 0;
                prevPageBtn.disabled = true;
                nextPageBtn.disabled = true;
                return;
            }}

            const startIndex = (currentPage - 1) * itemsPerPage;
            const endIndex = itemsPerPage === -1 ? filteredData.length : startIndex + itemsPerPage;
            const paginatedData = filteredData.slice(startIndex, endIndex);

            paginatedData.forEach(item => {{
                const row = tableBody.insertRow();
                // Use Object.keys(originalData[0]) to ensure column order based on the first row of data
                // Ensure originalData is not empty before accessing originalData[0]
                if (originalData.length > 0) {{
                    Object.keys(originalData[0]).forEach(key => {{
                        const cell = row.insertCell();
                        cell.textContent = item[key] === null ? '' : item[key]; // Handle null values
                    }});
                }} else {{
                    // Fallback if originalData somehow becomes empty here
                    Object.values(item).forEach(text => {{
                        const cell = row.insertCell();
                        cell.textContent = text === null ? '' : text;
                    }});
                }}
            }});

            updatePaginationControls();
        }}

        async function applyFilters() {{
            const globalSearch = globalFilter.value;
            const docId = docIdFilter.value;
            const discipline = disciplineFilter.value;
            const approvalStatus = approvalStatusFilter.value;
            const closedStatus = closedFilter.value;
            const overdueStatus = overdueFilter.value;

            // Call the Python backend filter function
            let sortedCol = currentSortColumn; // Default sort column
            if (!sortedCol && originalData.length > 0) {{
                sortedCol = Object.keys(originalData[0])[0];
            }} else if (originalData.length === 0) {{
                sortedCol = null;
            }}
            const sortDir = currentSortDirection || 'asc';

            // Only invoke filter_data if originalData is not empty, otherwise set filteredData to empty
            if (originalData.length > 0) {{
                const filteredJson = await google.colab.kernel.invokeFunction(
                    'filter_data',
                    [globalSearch, docId, discipline, approvalStatus, closedStatus, overdueStatus, sortedCol, sortDir]
                );
                filteredData = JSON.parse(filteredJson);
            }} else {{
                filteredData = [];
            }}

            currentPage = 1;
            renderTable();
        }}

        async function sortTable(column, direction = null, toggle = true) {{
            if (originalData.length === 0) return; // No data to sort

            if (toggle) {{
                if (currentSortColumn === column) {{
                    currentSortDirection = currentSortDirection === 'asc' ? 'desc' : 'asc';
                }} else {{
                    currentSortColumn = column;
                    currentSortDirection = 'asc';
                }}
            }}
            else if (direction) {{
                currentSortDirection = direction;
            }}

            // Re-apply filters which will also re-sort via backend
            await applyFilters(); // This will call filter_data with the new sort parameters
                                // and then renderTable
            // For now, it remains in sortTable, but ideally it should be after filteredData is finalized.
            document.querySelectorAll('#tableHeader th').forEach(th => {{
                th.classList.remove('sort-asc', 'sort-desc');
                if (th.getAttribute('data-column') === currentSortColumn) {{
                    th.classList.add('sort-' + currentSortDirection);
                }}
            }});
        }}

        function updatePaginationControls() {{
            const totalPages = itemsPerPage === -1 ? 1 : Math.ceil(filteredData.length / itemsPerPage);
            currentPageSpan.textContent = currentPage;
            totalPagesSpan.textContent = totalPages;

            prevPageBtn.disabled = currentPage === 1;
            nextPageBtn.disabled = currentPage === totalPages || filteredData.length === 0 || itemsPerPage === -1;
        }}

        // Event Listeners
        globalFilter.addEventListener('keyup', applyFilters);
        docIdFilter.addEventListener('change', applyFilters);
        disciplineFilter.addEventListener('change', applyFilters);
        approvalStatusFilter.addEventListener('change', applyFilters);
        closedFilter.addEventListener('change', applyFilters);
        overdueFilter.addEventListener('change', applyFilters);
        itemsPerPageSelect.addEventListener('change', () => {{
            itemsPerPage = parseInt(itemsPerPageSelect.value);
            currentPage = 1;
            renderTable();
        }});
        duckdbFileSelect.addEventListener('change', loadSelectedDuckDBFile); // New event listener for file selection

        // Initial setup
        initializePage(); // Call async initialization function

    </script>
</body>
</html>
"""

# Display the HTML in the Colab output
display(HTML(HTML_TEMPLATE))

# Load Excel file to create standalone interactive page

In [ ]:
%%html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Interactive Excel Viewer</title>
    <script src="https://cdn.jsdelivr.net/npm/xlsx@0.18.5/dist/xlsx.full.min.min.js"></script>
    <style>
        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; padding: 20px; background-color: #f4f7f6; color: #333; }
        .container { max-width: 1000px; margin: 0 auto; background: white; padding: 30px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
        .controls { display: flex; flex-wrap: wrap; gap: 15px; margin-top: 20px; padding: 15px; background: #f9f9f9; border-radius: 5px; }
        select { padding: 8px; border-radius: 4px; border: 1px solid #ccc; min-width: 150px; }
        table { width: 100%; border-collapse: collapse; margin-top: 20px; }
        th, td { border: 1px solid #ddd; padding: 12px; text-align: left; }
        th { background-color: #007bff; color: white; position: sticky; top: 0; }
        tr:nth-child(even) { background-color: #f2f2f2; }
        .hidden { display: none; }
        label { font-weight: bold; display: block; margin-bottom: 5px; font-size: 0.9em; }
    </style>
</head>
<body>

<div class="container">
    <h2>Excel Interactive Filter</h2>
    <p>Upload an Excel file (.xlsx or .xls) to generate dynamic filters based on your headers.</p>

    <input type="file" id="upload" accept=".xlsx, .xls" />

    <div id="filterContainer" class="controls hidden">
        </div>

    <div id="tableWrapper" class="hidden">
        <table id="dataTable">
            <thead id="tableHead"></thead>
            <tbody id="tableBody"></tbody>
        </table>
    </div>
</div>

<script>
    let jsonData = [];
    let headers = [];

    document.getElementById('upload').addEventListener('change', (e) => {
        const file = e.target.files[0];
        if (!file) return;

        const reader = new FileReader();
        reader.onload = (event) => {
            const data = new Uint8Array(event.target.result);
            const workbook = XLSX.read(data, { type: 'array' });

            // Get first sheet
            const firstSheetName = workbook.SheetNames[0];
            const worksheet = workbook.Sheets[firstSheetName];

            // Convert to JSON (first row as header)
            jsonData = XLSX.utils.sheet_to_json(worksheet);
            headers = Object.keys(jsonData[0]);

            renderFilters();
            renderTable(jsonData);

            document.getElementById('filterContainer').classList.remove('hidden');
            document.getElementById('tableWrapper').classList.remove('hidden');
        };
        reader.readAsArrayBuffer(file);
    });

    function renderFilters() {
        const container = document.getElementById('filterContainer');
        container.innerHTML = ''; // Clear existing

        headers.forEach(header => {
            const wrapper = document.createElement('div');
            const label = document.createElement('label');
            label.innerText = header;

            const select = document.createElement('select');
            select.id = `filter-${header}`;
            select.innerHTML = `<option value="">All</option>`;

            // Get unique values for this column
            const uniqueValues = [...new Set(jsonData.map(item => item[header]))];
            uniqueValues.forEach(val => {
                if (val !== undefined && val !== null) {
                    const opt = document.createElement('option');
                    opt.value = val;
                    opt.innerText = val;
                    select.appendChild(opt);
                }
            });

            select.addEventListener('change', applyFilters);
            wrapper.appendChild(label);
            wrapper.appendChild(select);
            container.appendChild(wrapper);
        });
    }

    function applyFilters() {
        let filtered = jsonData;

        headers.forEach(header => {
            const val = document.getElementById(`filter-${header}`).value;
            if (val !== "") {
                filtered = filtered.filter(row => String(row[header]) === val);
            }
        });

        renderTable(filtered);
    }

    function renderTable(data) {
        const head = document.getElementById('tableHead');
        const body = document.getElementById('tableBody');

        // Render Head
        head.innerHTML = `<tr>${headers.map(h => `<th>${h}</th>`).join('')}</tr>`;

        // Render Body
        body.innerHTML = data.map(row => {
            return `<tr>${headers.map(h => `<td>${row[h] || ''}</td>`).join('')}</tr>`;
        }).join('');
    }
</script>

</body>
</html>